In [0]:
%sql
-- Create the Medallion schemas inside dedicated catalog
CREATE SCHEMA IF NOT EXISTS maritime_ais.maritime_bronze 
MANAGED LOCATION 'abfss://maritime-lake@maritimepipeline.dfs.core.windows.net/bronze';

CREATE SCHEMA IF NOT EXISTS maritime_ais.maritime_silver 
MANAGED LOCATION 'abfss://maritime-lake@maritimepipeline.dfs.core.windows.net/silver';

CREATE SCHEMA IF NOT EXISTS maritime_ais.maritime_gold 
MANAGED LOCATION 'abfss://maritime-lake@maritimepipeline.dfs.core.windows.net/gold';

In [0]:
dbutils.widgets.text("datasets", "ais_locations,port_calls,sea_state", "Datasets to process (comma-separated)")
selected_datasets = [d.strip() for d in dbutils.widgets.get("datasets").split(",")]

from pyspark.sql.functions import current_timestamp, col

storage_account = "maritimepipeline"
container = "maritime-lake"
catalog_name = "maritime_ais"
schema_name = "maritime_bronze"
base_path = f"abfss://{container}@{storage_account}.dfs.core.windows.net"

for dataset in selected_datasets:
    print(f"Starting Auto Loader for: {dataset}...")

    landing_path = f"{base_path}/landing/{dataset}/"
    checkpoint_path = f"{base_path}/checkpoints/bronze_{dataset}/"
    schema_path = f"{base_path}/schemas/bronze_{dataset}/"
    table_name = f"{catalog_name}.{schema_name}.{dataset}"

    df_stream = (
        spark.readStream
        .format("cloudFiles")
        .option("cloudFiles.format", "json")
        .option("multiline", "true")
        .option("cloudFiles.schemaLocation", schema_path)
        .option("cloudFiles.inferColumnTypes", "true")
        .option("cloudFiles.schemaEvolutionMode", "addNewColumns")
        .load(landing_path)
    )

    df_enriched = (
        df_stream
        .withColumn("ingested_at", current_timestamp())
        .withColumn("source_file", col("_metadata.file_path"))
    )

    query = (
        df_enriched.writeStream
        .format("delta")
        .outputMode("append")
        .option("checkpointLocation", checkpoint_path)
        .option("mergeSchema", "true")
        .trigger(availableNow=True)
        .toTable(table_name)
    )

    query.awaitTermination()
    print(f"Finished processing {dataset}. Data appended to {table_name}.\n")

print(f"Bronze ingestion complete for: {selected_datasets}")